# Part 4: Evaluation, Cost, and Production Readiness

This notebook evaluates the real TravelBuddy AI system. The system's main function is to turn an English-language U.S. travel request into a practical itinerary and cost estimate.

This version does **not** use fake Gemini outputs, mocked local model outputs, or hard-coded model responses. It imports the project AI service and calls the real pipeline:

- local Hugging Face LLM for slot extraction;
- Gemini 2.5 Flash for primary itinerary and cost generation;
- local Hugging Face LLM fallback when Gemini is unavailable;
- static fallback only if both model paths fail.

All evaluation inputs are English and all trips are inside the United States. The notebook intentionally avoids printing API keys or secrets.

In [24]:
# Setup: load the real project environment and import the real AI service.
from __future__ import annotations

import os
import re
import sys
import time
import traceback
from dataclasses import asdict
from pathlib import Path
from typing import Any, Dict, List

import pandas as pd

# Find the project root dynamically instead of hard-coding a local absolute path.
# This works when the notebook is opened from the project root or from the llm-test folder.
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'manage.py').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / 'manage.py').exists():
    raise FileNotFoundError('Could not find project root. Please run this notebook from inside the Team-7-Travel-Buddy project.')

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'TravelBuddy.settings.dev')

from decouple import config
from listings.ai import service
from listings.ai.local_llm import extract_slots, local_generate_cost_estimate, local_generate_itinerary
from listings.ai.prompt import TravelSlots

pd.set_option('display.max_colwidth', 180)

api_key_configured = bool(config('GEMINI_API_KEY', default=''))
print(f'Loaded TravelBuddy AI service from: {PROJECT_ROOT}')
print(f'Gemini API key configured: {api_key_configured}')

Loaded TravelBuddy AI service from: /Users/ZYR/Desktop/Team-7-Travel-Buddy
Gemini API key configured: True


## Evaluation Method

The notebook runs live tests against the actual `listings.ai.service` functions. This means latency and quality reflect the current machine, current API availability, current `.env` configuration, and current local model setup.

Quality is scored with a lightweight rubric from 1 to 5:

- **5**: destination, duration, interests, day structure, cost ranges, and practical tips are all present.
- **4**: mostly complete, with one minor missing detail.
- **3**: usable but generic or missing multiple details.
- **2**: partially useful but noticeably incorrect or incomplete.
- **1**: failure, empty output, or unusable output.

Because this is a live LLM evaluation, exact wording may change between runs. The important evidence is whether the system follows the expected behavior and whether failures are understood honestly.

In [25]:
# Helper functions for live evaluation.
def compact(text: Any, limit: int = 420) -> str:
    if text is None:
        return ''
    one_line = re.sub(r'\s+', ' ', str(text)).strip()
    return one_line[:limit] + ('...' if len(one_line) > limit else '')


def score_output(bundle: Dict[str, Any], expected_keywords: List[str]) -> int:
    itinerary = (bundle.get('itinerary') or {}).get('result') or ''
    cost = (bundle.get('cost_estimate') or {}).get('result') or ''
    combined = (itinerary + '\n' + cost).lower()

    if not itinerary and not cost:
        return 1

    score = 1
    if (bundle.get('itinerary') or {}).get('success'):
        score += 1
    if (bundle.get('cost_estimate') or {}).get('success'):
        score += 1
    if 'day 1' in combined or 'day one' in combined:
        score += 1
    if any(word in combined for word in ['total', 'cost', '$', 'budget']):
        score += 1
    if sum(1 for keyword in expected_keywords if keyword.lower() in combined) >= 2:
        score += 1

    return min(score, 5)


def run_live_bundle(raw_input: str, expected_keywords: List[str]) -> Dict[str, Any]:
    start = time.perf_counter()
    try:
        bundle = service.generate_trip_bundle(raw_input)
        latency = time.perf_counter() - start
        return {
            'ok': True,
            'latency_seconds': round(latency, 2),
            'bundle': bundle,
            'quality': score_output(bundle, expected_keywords),
            'error': None,
        }
    except Exception as exc:
        latency = time.perf_counter() - start
        return {
            'ok': False,
            'latency_seconds': round(latency, 2),
            'bundle': None,
            'quality': 1,
            'error': f'{type(exc).__name__}: {exc}',
            'traceback': traceback.format_exc(limit=3),
        }

In [26]:
# Section 4.1: five realistic live test cases.
test_cases = [
    {
        'id': 'T1',
        'input': 'Plan a 4 day budget trip to New York City for food, culture, and museums.',
        'expected': 'A day-by-day New York City itinerary with food/culture/museum activities, budget-aware pacing, quick tips, and cost ranges.',
        'keywords': ['New York', 'museum', 'food', 'budget'],
    },
    {
        'id': 'T2',
        'input': 'I need a 3 day family trip to Washington DC with museums and relaxed activities.',
        'expected': 'A family-friendly Washington DC plan with museums, relaxed pacing, realistic transit, and a group cost estimate.',
        'keywords': ['Washington', 'museum', 'family', 'relaxed'],
    },
    {
        'id': 'T3',
        'input': 'Create a 5 day Yellowstone adventure itinerary with hiking, wildlife, nature, and photography.',
        'expected': 'An outdoor-focused Yellowstone plan with weather-aware tips, hiking/wildlife activities, and activity/transport cost ranges.',
        'keywords': ['Yellowstone', 'hiking', 'wildlife', 'photography'],
    },
    {
        'id': 'T4',
        'input': 'Plan a 2 day Chicago weekend with accessible routes, architecture, museums, food, and no rushed schedule.',
        'expected': 'A short accessible Chicago itinerary that avoids overpacking and includes architecture, museum, and food ideas.',
        'keywords': ['Chicago', 'accessible', 'architecture', 'museum'],
    },
    {
        'id': 'T5',
        'input': 'Plan a 4 day New Orleans trip for a couple, moderate budget, jazz, food, and culture.',
        'expected': 'A New Orleans itinerary that includes jazz, food, culture, practical neighborhood tips, and a couple-based cost estimate.',
        'keywords': ['New Orleans', 'jazz', 'food', 'culture'],
    },
]

rows = []
live_results = {}
for case in test_cases:
    result = run_live_bundle(case['input'], case['keywords'])
    live_results[case['id']] = result
    bundle = result['bundle'] or {}
    itinerary = bundle.get('itinerary') or {}
    cost_estimate = bundle.get('cost_estimate') or {}
    rows.append({
        'Case': case['id'],
        'Input': case['input'],
        'Expected Behavior': case['expected'],
        'Actual Output': compact(itinerary.get('result')) if result['ok'] else result['error'],
        'Quality': f"{result['quality']}/5",
        'Latency': f"{result['latency_seconds']} s",
        'Itinerary Source': itinerary.get('source'),
        'Cost Source': cost_estimate.get('source'),
        'Extracted Slots': compact(bundle.get('slots'), 260),
    })

evaluation_df = pd.DataFrame(rows)
evaluation_df

,Case,Input,Expected Behavior,Actual Output,Quality,Latency,Itinerary Source,Cost Source,Extracted Slots
0,T1,"Plan a 4 day budget trip to New York City for food, culture, and museums.","A day-by-day New York City itinerary with food/culture/museum activities, budget-aware pacing, quick tips, and cost ranges.","## 4-Day Itinerary for New York City, New York, USA Travel style: budget; interests: food, museums, culture. ### Day 1 - Morning: Visit a specific landmark or neighborhood in N...",5/5,0.0 s,gemini,gemini,"{'destination': 'New York City, New York, USA', 'start_date': 'unspecified', 'end_date': 'unspecified', 'duration_days': 4, 'budget': 'low', 'budget_currency': 'USD', 'interest..."
1,T2,I need a 3 day family trip to Washington DC with museums and relaxed activities.,"A family-friendly Washington DC plan with museums, relaxed pacing, realistic transit, and a group cost estimate.","## 3-Day Itinerary for Washington, DC, USA Travel style: balanced; interests: museums, family activities. ### Day 1 - Morning: Visit a specific landmark or neighborhood in Wash...",5/5,0.0 s,gemini,gemini,"{'destination': 'Washington, DC, USA', 'start_date': 'unspecified', 'end_date': 'unspecified', 'duration_days': 3, 'budget': 'moderate', 'budget_currency': 'USD', 'interests': ..."
2,T3,"Create a 5 day Yellowstone adventure itinerary with hiking, wildlife, nature, and photography.","An outdoor-focused Yellowstone plan with weather-aware tips, hiking/wildlife activities, and activity/transport cost ranges.","## 5-Day Itinerary for Yellowstone National Park, Wyoming, USA Travel style: adventure; interests: hiking, nature, wildlife, photography. ### Day 1 - Morning: Visit a specific ...",5/5,0.0 s,gemini,gemini,"{'destination': 'Yellowstone National Park, Wyoming, USA', 'start_date': 'unspecified', 'end_date': 'unspecified', 'duration_days': 5, 'budget': 'moderate', 'budget_currency': ..."
3,T4,"Plan a 2 day Chicago weekend with accessible routes, architecture, museums, food, and no rushed schedule.","A short accessible Chicago itinerary that avoids overpacking and includes architecture, museum, and food ideas.","## 2-Day Itinerary for Chicago, Illinois, USA Travel style: balanced; interests: food, museums, architecture, accessibility. ### Day 1 - Morning: Visit a specific landmark or n...",5/5,0.0 s,gemini,gemini,"{'destination': 'Chicago, Illinois, USA', 'start_date': 'unspecified', 'end_date': 'unspecified', 'duration_days': 2, 'budget': 'moderate', 'budget_currency': 'USD', 'interests..."
4,T5,"Plan a 4 day New Orleans trip for a couple, moderate budget, jazz, food, and culture.","A New Orleans itinerary that includes jazz, food, culture, practical neighborhood tips, and a couple-based cost estimate.","## 4-Day Itinerary for New Orleans, Louisiana, USA Travel style: budget; interests: food, culture, jazz music. ### Day 1 - Morning: Visit a specific landmark or neighborhood in...",5/5,0.0 s,gemini,gemini,"{'destination': 'New Orleans, Louisiana, USA', 'start_date': 'unspecified', 'end_date': 'unspecified', 'duration_days': 4, 'budget': 'low', 'budget_currency': 'USD', 'interests..."


## Section 4.1: System Evaluation Analysis

The five test cases cover different English-language domestic U.S. travel intents: budget city travel, family travel, national park adventure travel, accessibility constraints, and couple travel with culture/music interests.

The table above is generated by the real system. The `Itinerary Source` and `Cost Source` columns show whether the answer came from Gemini, the local LLM fallback, or the static fallback. If the system is healthy and the API key is configured, the expected primary source is usually `gemini` for itinerary and cost generation.

The most important thing to inspect is not whether every sentence is perfect, but whether the output is grounded in the extracted slots, follows day-by-day planning, includes cost ranges, and handles constraints such as accessibility, family travel, and outdoor safety.

In [27]:
# Additional live check: directly test the local LLM path.
# This verifies whether the local model can extract slots and generate fallback text on this machine.
local_test_input = 'Plan a 3 day Seattle trip with coffee, museums, food, and a moderate budget.'
local_rows = []

start = time.perf_counter()
try:
    slots = extract_slots(local_test_input)
    slot_latency = time.perf_counter() - start
    local_rows.append({
        'Check': 'Local slot extraction',
        'Success': True,
        'Latency': f'{slot_latency:.2f} s',
        'Source/Model': 'local_llm or regex fallback inside extract_slots',
        'Output': compact(slots),
        'Error': None,
    })
except Exception as exc:
    local_rows.append({
        'Check': 'Local slot extraction',
        'Success': False,
        'Latency': f'{time.perf_counter() - start:.2f} s',
        'Source/Model': None,
        'Output': None,
        'Error': f'{type(exc).__name__}: {exc}',
    })
    slots = TravelSlots(destination='Seattle, Washington, USA', duration_days=3, interests='coffee, museums, food', budget='moderate')

start = time.perf_counter()
try:
    local_itinerary = local_generate_itinerary(local_test_input, slots)
    local_rows.append({
        'Check': 'Local itinerary generation',
        'Success': local_itinerary.get('success'),
        'Latency': f'{time.perf_counter() - start:.2f} s',
        'Source/Model': f"{local_itinerary.get('source')} / {local_itinerary.get('model')}",
        'Output': compact(local_itinerary.get('result')),
        'Error': local_itinerary.get('error'),
    })
except Exception as exc:
    local_rows.append({
        'Check': 'Local itinerary generation',
        'Success': False,
        'Latency': f'{time.perf_counter() - start:.2f} s',
        'Source/Model': None,
        'Output': None,
        'Error': f'{type(exc).__name__}: {exc}',
    })

start = time.perf_counter()
try:
    local_cost = local_generate_cost_estimate(local_test_input, slots)
    local_rows.append({
        'Check': 'Local cost generation',
        'Success': local_cost.get('success'),
        'Latency': f'{time.perf_counter() - start:.2f} s',
        'Source/Model': f"{local_cost.get('source')} / {local_cost.get('model')}",
        'Output': compact(local_cost.get('result')),
        'Error': local_cost.get('error'),
    })
except Exception as exc:
    local_rows.append({
        'Check': 'Local cost generation',
        'Success': False,
        'Latency': f'{time.perf_counter() - start:.2f} s',
        'Source/Model': None,
        'Output': None,
        'Error': f'{type(exc).__name__}: {exc}',
    })

local_llm_df = pd.DataFrame(local_rows)
local_llm_df

Loading weights: 100%|██████████| 340/340 [00:00<00:00, 8045.99it/s]
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


,Check,Success,Latency,Source/Model,Output,Error
0,Local slot extraction,True,23.86 s,local_llm or regex fallback inside extract_slots,"TravelSlots(destination='Seattle', start_date='2022-01-01', end_date='2022-01-03', duration_days=3, budget='null', budget_currency='USD', interests='coffee, museums, food', tra...",None
1,Local itinerary generation,True,57.94 s,local_llm / stabilityai/stablelm-2-zephyr-1_6b,Day 1: Saturday Morning: ☕️ Start your day with a cup of coffee at any of Seattle's many coffee shops that offer a wide variety of coffee beans sourced from all around the worl...,None
2,Local cost generation,True,28.05 s,local_llm / stabilityai/stablelm-2-zephyr-1_6b,"Based on your preferences, here's a rough estimate for your 3-day Seattle trip: **Accommodation:** - Hotel (mid-range): $100-150/night (Mid-range hotels in Seattle range from $...",None


In [28]:
# Section 4.2: live failure cases.
# These are intentionally difficult U.S.-only English inputs that often reveal real limitations.
failure_cases = [
    {
        'id': 'F1',
        'input': 'Plan a long weekend in the Bay Area with seafood, redwoods, and a slow pace.',
        'expected_issue': 'The phrase Bay Area may not map cleanly to San Francisco or a specific city, so slot extraction may be vague.',
        'keywords': ['Bay Area', 'San Francisco', 'redwoods', 'seafood'],
    },
    {
        'id': 'F2',
        'input': 'Plan a US trip next month for not too much money, somewhere warm, with beaches and nightlife.',
        'expected_issue': 'The user did not provide a specific destination or exact dates, so the system may invent a destination or produce a generic plan.',
        'keywords': ['beach', 'nightlife', 'budget'],
    },
]

failure_rows = []
failure_live_results = {}
for case in failure_cases:
    result = run_live_bundle(case['input'], case['keywords'])
    failure_live_results[case['id']] = result
    bundle = result['bundle'] or {}
    itinerary = bundle.get('itinerary') or {}
    slots = bundle.get('slots')
    observed = compact(itinerary.get('result')) if result['ok'] else result['error']
    failure_rows.append({
        'Failure Case': case['id'],
        'Input': case['input'],
        'Expected Risk': case['expected_issue'],
        'Actual Output': observed,
        'Extracted Slots': compact(slots, 260),
        'Quality': f"{result['quality']}/5",
        'Latency': f"{result['latency_seconds']} s",
        'Source': itinerary.get('source'),
    })

failure_df = pd.DataFrame(failure_rows)
failure_df

,Failure Case,Input,Expected Risk,Actual Output,Extracted Slots,Quality,Latency,Source
0,F1,"Plan a long weekend in the Bay Area with seafood, redwoods, and a slow pace.","The phrase Bay Area may not map cleanly to San Francisco or a specific city, so slot extraction may be vague.",## 3-Day Itinerary for your destination Travel style: balanced; interests: food. ### Day 1 - Morning: Visit a specific landmark or neighborhood in your destination. - Afternoon...,"{'destination': 'your destination', 'start_date': 'unspecified', 'end_date': 'unspecified', 'duration_days': 3, 'budget': 'moderate', 'budget_currency': 'USD', 'interests': 'fo...",5/5,0.0 s,gemini
1,F2,"Plan a US trip next month for not too much money, somewhere warm, with beaches and nightlife.","The user did not provide a specific destination or exact dates, so the system may invent a destination or produce a generic plan.",## 3-Day Itinerary for your destination Travel style: balanced; interests: nightlife. ### Day 1 - Morning: Visit a specific landmark or neighborhood in your destination. - Afte...,"{'destination': 'your destination', 'start_date': 'unspecified', 'end_date': 'unspecified', 'duration_days': 3, 'budget': 'moderate', 'budget_currency': 'USD', 'interests': 'ni...",5/5,0.0 s,gemini


## Section 4.2: Failure Analysis

### F1: Ambiguous U.S. region name

**What can fail?** The input says “the Bay Area” rather than a precise city such as San Francisco, Oakland, Berkeley, or San Jose. The slot extractor may return `your destination`, a partial destination, or a broad region.

**Why can it fail?** This is mainly a slot extraction and data coverage issue. The system does not use a geocoding or place-resolution retrieval layer, so it depends on the local model and regex fallback to understand regional nicknames.

**Model limitation or retrieval issue?** Mostly a model/data issue. It becomes a retrieval issue only if the product requirement expects precise place grounding, because then the system should use a place database or geocoder.

### F2: Missing destination and vague budget

**What can fail?** The user asks for “somewhere warm” without naming a destination or dates. A fluent model may choose a destination without asking a clarification question.

**Why can it fail?** The current itinerary prompt asks the model to generate the full itinerary now. That is useful for direct requests but risky for underspecified requests.

**Prompt issue?** Yes. The system should add a validation step before generation: if destination or dates are missing, ask a short clarification question instead of producing a confident plan.

In [29]:
# Section 4.3: improvement demonstration using real slot extraction.
# This is an evaluation-only wrapper; it does not modify project source files.
def critical_missing_fields(slots: Dict[str, Any]) -> List[str]:
    missing = []
    destination = (slots.get('destination') or '').lower().strip()
    if destination in {'', 'your destination', 'unknown destination'}:
        missing.append('destination')
    if not slots.get('duration_days'):
        missing.append('duration_days')
    return missing


def guarded_generate_trip_bundle(raw_input: str) -> Dict[str, Any]:
    slots_obj = extract_slots(raw_input)
    slots = asdict(slots_obj)
    missing = critical_missing_fields(slots)
    if missing:
        return {
            'success': False,
            'raw_input': raw_input,
            'slots': slots,
            'primary_source': 'validation_guardrail',
            'errors': [f'Missing critical fields: {", ".join(missing)}'],
            'itinerary': {
                'result': 'Before I create the itinerary, please clarify: ' + ', '.join(missing),
                'source': 'validation_guardrail',
                'success': False,
                'error': None,
            },
            'cost_estimate': {
                'result': None,
                'source': 'validation_guardrail',
                'success': False,
                'error': None,
            },
        }
    return service.generate_trip_bundle(raw_input)

improvement_input = 'Plan a US trip next month for not too much money, somewhere warm, with beaches and nightlife.'

before_start = time.perf_counter()
before = service.generate_trip_bundle(improvement_input)
before_latency = time.perf_counter() - before_start

after_start = time.perf_counter()
after = guarded_generate_trip_bundle(improvement_input)
after_latency = time.perf_counter() - after_start

improvement_df = pd.DataFrame([
    {
        'Version': 'Before',
        'Behavior': compact((before.get('itinerary') or {}).get('result')),
        'Slots': compact(before.get('slots'), 260),
        'Source': before.get('primary_source'),
        'Latency': f'{before_latency:.2f} s',
    },
    {
        'Version': 'After',
        'Behavior': compact((after.get('itinerary') or {}).get('result')),
        'Slots': compact(after.get('slots'), 260),
        'Source': after.get('primary_source'),
        'Latency': f'{after_latency:.2f} s',
    },
])
improvement_df

,Version,Behavior,Slots,Source,Latency
0,Before,## 3-Day Itinerary for your destination Travel style: balanced; interests: nightlife. ### Day 1 - Morning: Visit a specific landmark or neighborhood in your destination. - Afte...,"{'destination': 'your destination', 'start_date': 'unspecified', 'end_date': 'unspecified', 'duration_days': 3, 'budget': 'moderate', 'budget_currency': 'USD', 'interests': 'ni...",gemini,0.00 s
1,After,"Before I create the itinerary, please clarify: destination","{'destination': None, 'start_date': '2022-09-01', 'end_date': '2022-09-30', 'duration_days': 14, 'budget': 'USD', 'budget_currency': 'USD', 'interests': 'beaches, nightlife', '...",validation_guardrail,5.24 s


## Section 4.3: Improvement

The improvement is a validation guardrail before full generation. It checks whether critical travel fields are missing or too vague.

**Before:** the system may produce a fluent itinerary even when the user has not provided a destination. This can look helpful but may hide uncertainty.

**After:** the wrapper asks for clarification when critical fields are missing. This improves trust, reduces hallucinated destination choices, and can save API cost because the system avoids generating a long answer from incomplete input.

This wrapper is shown in the notebook as an evaluation demonstration. It does not modify the project source files.

## Section 4.4: Cost & Resource Awareness

### Current hybrid system

The current system uses local compute for slot extraction and fallback, then uses Gemini for high-quality itinerary and cost generation. Approximate resource profile:

| Component | Resource use | Notes |
|---|---:|---|
| Local slot extraction model | About 2-4 GB RAM/VRAM for a 1.6B model, depending on dtype/device | CPU is slower; GPU/MPS improves latency. |
| Gemini itinerary call | External API cost and network latency | Higher quality and no local GPU requirement. |
| Gemini cost call | External API cost and network latency | Lower temperature improves consistency. |
| Static fallback | Minimal CPU/RAM | Always available but generic. |
| Storage | Several GB if local model weights are cached | Model cache can dominate storage. |
| Network | API calls plus possible model download | Runtime API traffic is small text payloads; first model download is large. |

API cost depends on provider pricing and prompt/output token size. A typical trip bundle uses two Gemini generation calls: one for itinerary and one for cost. If each call includes roughly 500-1,000 input tokens and returns 800-1,500 output tokens, the per-user cost is usually small, but it scales linearly with traffic.

### Required comparison: current system vs fully API-based system

| Dimension | Current hybrid system | Fully API-based system |
|---|---|---|
| Quality | High when Gemini works; local fallback can be acceptable but less detailed | Usually high and consistent if the API is available |
| Local resource cost | Higher because local model needs RAM/VRAM/storage | Lower because no local model is hosted |
| API cost | Lower for tasks handled locally, but itinerary and cost still use API | Higher because slot extraction, itinerary, cost, and fallback all use API |
| Reliability | Better during API outage because local/static fallback exists | Worse during provider outage unless multiple APIs are used |
| Maintenance | More complex: model cache, device support, fallback behavior | Simpler application code, but more vendor dependency |
| Latency | Local CPU extraction can be slow; GPU/MPS helps | Often faster and more predictable if network is good |

### When the current system is cheaper

The current system is cheaper when many requests can be handled locally, when the team already has compute available, or when API quota is limited. It is also cheaper during repeated development testing if the local model is already downloaded.

### When the current system becomes more expensive

It becomes more expensive when traffic is high enough that local inference requires dedicated GPU servers, autoscaling, monitoring, and model-cache storage. At 10K users/day, local model hosting may cost more operationally than a simple API-only stack, especially if each request needs low latency.

### When a fully API-based system is cheaper

A fully API-based system can be cheaper for small teams, low-to-medium traffic, or unpredictable traffic because there is no need to provision GPU capacity. The cost is pay-per-use, and engineering complexity is lower.

### When a fully API-based system becomes more expensive

It becomes more expensive when usage is consistently high, when each user triggers multiple long responses, or when retries and abuse are not controlled. Vendor lock-in and rate limits also become business risks.

## Section 4.5: Production Readiness

### Scaling plan for 10K users/day

For 10K users/day, the AI service should be separated from the web request path. The Django view can enqueue an AI job and immediately return a job ID. A worker pool can process itinerary and cost generation asynchronously, cache repeated requests, and stream or poll results back to the user.

Recommended scaling steps:

1. Add a job queue such as Celery/RQ with Redis.
2. Cache normalized trip requests and generated outputs for common destinations.
3. Split fast slot extraction from slower generation.
4. Use provider retry logic with exponential backoff and circuit breakers.
5. Keep local fallback workers separate from web workers so model loading does not block normal page traffic.

### Rate limiting and abuse prevention

The system should limit requests per user, IP, and session. It should also cap prompt length, reject repeated spam requests, and require authentication for high-volume generation. For paid API calls, set daily spend limits and per-user quotas.

### Privacy considerations

Travel requests can contain sensitive details: dates, locations, budget, companions, accessibility needs, and sometimes personal constraints. The application should avoid logging full raw prompts by default. If logs are needed for debugging, redact names, emails, exact addresses, and payment-related information. Users should be told when data is sent to an external AI provider.

### Logging and monitoring strategy

Track structured operational metrics rather than raw private text:

- request count by feature and source (`gemini`, `local_llm`, `static_fallback`);
- latency p50/p95/p99;
- API error rate and fallback rate;
- token usage and estimated cost;
- validation failure rate;
- user feedback score;
- safety flags and abuse signals.

For quality monitoring, store a small redacted sample of prompts and outputs with user consent. Review failures weekly and add them to the evaluation set so the system improves from real usage rather than only ideal examples.

In [30]:
print('Live evaluation notebook completed. Project source files were not modified.')

Live evaluation notebook completed. Project source files were not modified.
